# Export Evo Geoscience objects to an AGS file

This notebook shows how you can login to Evo and export DownholeCollection objects from your Evo workspace to an AGS file.

**Requirements for export:**
The DownholeCollection must at minimum have data for AGS4 required fields:

- `LOCA` - Location Details (borehole/test locations)
- `SCPG` - Static Cone Penetration Tests - General
- `SCPT` - Seismic Cone Penetration Test results

In this first cell we create a ServiceManagerWidget which will open a browser window and ask you to login.

Once logged in, a widget will be displayed below allowing you to select an organisation and workspace to publish objects to.

__Required:__ The following cell requires credential configuration.

In [ ]:
%load_ext autoreload
%autoreload 2


import os
from pprint import pp

from evo.notebooks import ServiceManagerWidget
from evo.data_converters.common import create_evo_object_service_and_data_client

# Credentials can be provided from .env or filled into second params below.
# Use `uv run --env-file .env` to load environment variables from .env file.
client_id = os.getenv("EVO_CLIENT_ID", "")
base_uri = os.getenv("EVO_BASE_URI", "")
discovery_url = os.getenv("EVO_DISCOVERY_URL", "")

if not client_id or not base_uri or not discovery_url:
    raise ValueError("Evo credentials not provided.")

manager = await ServiceManagerWidget.with_auth_code(
    client_id=client_id, base_uri=base_uri, discovery_url=discovery_url
).login()

object_service_client, data_client = create_evo_object_service_and_data_client(service_manager_widget=manager)

In the cell below we choose the AGS file we want to publish and set its path in the `ags_file` variable.

You may also specify tags to add to the created Geoscience objects.

Then we call `convert_ags`, passing it the AGS file path, the service manager widget from above, a path we want the published objects to appear under, and finally whether we want existing objects to be overwritten.

The function will parse the AGS file and convert it into a Downhole Collection object, which will then be published to Evo.

Then we print out the metadata of the object that was published to Evo.

_Note:_ If the AGS file cannot be parsed or contains invalid data, an error message will be shown and no objects will be published.

In [ ]:
from packages.ags.src.evo.data_converters.ags.exporter.evo_to_ags import export_ags
from evo.data_converters.common.evo_client import EvoObjectMetadata
from evo_schemas.objects import DownholeCollection_V1_3_1

# This is the UUID of an Evo object uploaded using adjacent import_ags/import-ags.ipynb notebook.
object_uuid = "c283dcfb-a8ea-4f36-ae4f-6ff9efcf086f"
downloaded_object = await object_service_client.download_object_by_id(object_id=object_uuid)
pp(f"Downloaded object: {downloaded_object}")

# pp(downloaded_object.as_dict())

remote_dhc = DownholeCollection_V1_3_1.from_dict(downloaded_object.as_dict())

pp(remote_dhc.location.hole_id.table.as_dict())

collars_df = (await downloaded_object.download_dataframe(remote_dhc.location.hole_id.table.as_dict())).rename(
    columns={"key": "hole_index", "value": "hole_id"}
)

pp(collars_df)

In [ ]:
object_meta = EvoObjectMetadata(object_id=object_uuid)
downholes_to_export = [object_meta]
export_ags(filepath="export.ags", objects=downholes_to_export, service_manager_widget=manager)

In [ ]:
file_path = "export.ags"

try:
    with open(file_path, "r") as file:
        file_content = file.read()
        print(file_content)
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while reading the file: {e}")